# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as a single object; no subscripting or iteration)
metadata = dataset.metadata

# Print basic metadata info
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s. By listing record sets and their contained entities, you can plan further extraction and analysis.

Each record set, field, or column should always be referenced by its `@id`.

In [ ]:
# Fetch all available record sets and their IDs
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in this dataset schema.")
else:
    print("Available Record Sets and their @ids:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}", f"name: {rs.get('name', '<no name>')}")

    # For each record set, show the fields and columns by @id
    for rs in record_sets:
        print(f"\nRecord set '@id': {rs['@id']}\n  Fields (by @id):")
        fields = rs.get('field', [])
        for f in fields:
            print(f"    - {f['@id']} (name: {f.get('name', '<no name>')}, type: {f.get('dataType', '<no type>')})")
        columns = rs.get('column', [])
        if columns:
            print("  Columns (by @id):")
            for c in columns:
                print(f"    - {c['@id']} (name: {c.get('name', '<no name>')})")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Reference record set and field `@id`s directly from the overview above.

**Note:** The FAIR^2 dataset contains structured tabulations for clinical features, hormone levels, imaging, and genetic variants—each of these is likely a separate record set. All entities are referenced by their `@id`.

_For demonstration, we'll attempt to extract all available record sets._

In [ ]:
# List all record set @ids for extraction

record_set_ids = []
if metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in metadata.recordSet]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set '@id': {record_set_id}")
    try:
        # Extract records using mlcroissant (all references by @id)
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"  Unable to load data for record set {record_set_id}: {e}")

# If no record sets were found, display message
if not dataframes:
    print("No dataframes extracted. Please check the schema at the provided URL.")

## 4. Exploratory Data Analysis (EDA)

For each loaded record set DataFrame, apply processing and filtering steps. This includes selecting a numeric field, removing outliers, normalizing values, and grouping records using distinct columns—all referenced by their `@id`.

Choose relevant `@id`s from the overview. Below is a modeling code, adjust per the real column names for your dataset.

In [ ]:
# EDA for a selected record set
# Replace these @id values with the ones from your overview section.
if dataframes:
    # Select the first record set for demonstration
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]

    # Identify a numeric field, by @id
    # Example field id (replace with real one from your schema)
    numeric_field_id = None
    if not df.empty:
        numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float]]
        if numeric_candidates:
            numeric_field_id = numeric_candidates[0]
        else:
            print("No numeric field detected in this record set.")

    # Apply filtering: Filter records based on a threshold
    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # Example: get mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records: '{numeric_field_id}' > {threshold:.2f}")
        display(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Grouping by another field (pick a categorical field by @id)
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped by '{group_field_id}' and mean '{numeric_field_id}':")
            display(grouped_df.head())
        else:
            print("No group field found in this record set.")
    else:
        print("No numeric field available for EDA in selected record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize numeric data distributions and relationships, referencing fields by their `@id`. For demonstration, a histogram and a bar plot are provided.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    # Try to locate a numeric field for plotting
    numeric_fields = [col for col in df.columns if df[col].dtype in [int, float]]
    if numeric_fields:
        numeric = numeric_fields[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric], kde=True)
        plt.title(f"Distribution of '{numeric}'")
        plt.xlabel(numeric)
        plt.ylabel('Frequency')
        plt.show()

        # Try a categorical field
        categorical_fields = [col for col in df.columns if df[col].dtype == object]
        if categorical_fields:
            cat = categorical_fields[0]
            plt.figure(figsize=(8, 4))
            df.groupby(cat)[numeric].mean().plot(kind='bar')
            plt.title(f"Mean '{numeric}' by '{cat}'")
            plt.xlabel(cat)
            plt.ylabel(f"Mean {numeric}")
            plt.show()
    else:
        print("No numeric fields found for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library. Data was referenced entirely by `@id`, in accordance with Croissant schema standards. Record sets, fields, and columns were loaded and processed, including basic exploratory analysis and visualization.

Key points:
- Metadata and structure accessed via the Croissant schema URL.
- Record sets, fields, and columns referenced by their `@id` for consistency.
- Dataframes extracted for each record set available, with basic EDA and visualization demonstrated.

Please adjust numeric and categorical field selections per the actual column `@id`s from your schema overview for further, domain-specific analysis.